# Planars — {{LANG_ID}} Annotation Validation

Checks all annotation sheets for this language against their allowed parameter values.
Invalid cells are highlighted pink in Google Sheets so you can find and fix them easily.

Run all cells with **Runtime → Run all**. When prompted, sign in with your Google account.

In [ ]:
#@title Setup
import subprocess, sys, importlib

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'git+https://github.com/jcgood/planars.git',
     'gspread', 'google-auth'])
importlib.invalidate_caches()

from google.colab import auth
auth.authenticate_user()

import json, gspread
from google.auth import default
from googleapiclient.discovery import build

creds, _ = default(scopes=[
    'https://www.googleapis.com/auth/drive',
    'https://www.googleapis.com/auth/spreadsheets',
])
gc = gspread.authorize(creds)
drive_svc = build('drive', 'v3', credentials=creds)

# CONFIG_FILE_ID and LANG_ID are baked in at notebook generation time.
CONFIG_FILE_ID = '{{CONFIG_FILE_ID}}'
LANG_ID = '{{LANG_ID}}'

config = json.loads(drive_svc.files().get_media(fileId=CONFIG_FILE_ID).execute())
manifest = {LANG_ID: config[LANG_ID]}

print('Ready ✓')

In [ ]:
#@title Validate sheets and update highlighting
from coding.validate_coding import validate_annotation_rows, highlight_cells, clear_highlights

lang_data = manifest[LANG_ID]
sheets_info = lang_data.get('sheets', {})

total_issues = 0
results = []

for class_name, sheet_info in sorted(sheets_info.items()):
    sid = sheet_info.get('spreadsheet_id') or sheet_info.get('id')
    if not sid:
        continue
    try:
        ss = gc.open_by_key(sid)
    except Exception as e:
        print(f'[{class_name}] could not open: {e}')
        continue

    constructions = sheet_info.get('constructions', {})
    for ws in ss.worksheets():
        construction = ws.title
        param_info = constructions.get(construction, {})
        expected_params = param_info.get('params', [])
        param_values = param_info.get('param_values', {})

        rows = ws.get_all_values()
        _, issues = validate_annotation_rows(rows, expected_params, construction, param_values)
        bad_cells = [i.cell for i in issues if i.cell is not None]
        clear_highlights(ws)
        highlight_cells(ws, bad_cells)

        total_issues += len(issues)
        results.append((class_name, construction, issues))

print(f'Validation complete. {total_issues} issue(s) found across {len(results)} sheet(s).')
if total_issues:
    print('Pink cells updated in your Google Sheets.')
else:
    print('All clear – no issues found. Any previous pink highlights have been cleared.')

In [ ]:
#@title Issue summary
if not any(issues for _, _, issues in results):
    print('No issues.')
else:
    for class_name, construction, issues in results:
        if not issues:
            continue
        print(f'\n{class_name} / {construction} ({len(issues)} issue(s)):')
        for issue in issues:
            print(f'  {issue}')